# T03-A — Data quality, duplicate, and causal-integrity audit

**Authorized scope:** pre-split implementation and evidence only. This notebook does not construct the T05 split, access held-out membership or labels, run the 2,000-draw calibration, train outcome/uplift models, remove rows, or accept T03.

Research flow: **Question → Method → Evidence → Interpretation → What cannot be concluded → Required/deferred action.**

## 1. Purpose, authority, and frozen decisions

Authority order: `decision_register.csv` → frozen contracts/protocols/accepted ADR decisions → accepted T01/T02 evidence → Issue #4 → implementation.

- T03-D01: inherited evidence taxonomy.
- T03-D02: accepted `CONDITIONAL_PERMUTATION_APPROXIMATION`.
- T03-D03: retain every released row in the primary analysis.
- T03-D04: normalized run-scoped serialization.
- `X=f0…f11`, `T=treatment`, `Y=conversion`; `visit` is secondary, `exposure` audit-only, and `_source_row_id` provenance-only.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
import psutil
import pyarrow as pa

def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    FEATURE_COLUMNS, PROCESSED_COLUMNS, RAW_SCHEMA, SOURCE_ROW_ID,
    finalize_artifact_manifest, implementation_environment, load_selector,
    materialize_pandas, open_csv_reader, open_processed_dataset, portable_repo_path, sha256_file,
)
from src.audit import (
    AuditContractError, algorithm_contract, balance_diagnostics,
    calibration_artifact_schema, canonical_block_membership_hash,
    conditional_permute_treatment, cross_fitted_treatment_predictability,
    deferred_evidence_record, duplicate_audit, pending_disposition_evidence_record,
    precision_reconciliation, replication_seed_metadata, validate_pre_split_frame,
    write_t03_csv_new, write_t03_json_new,
)

NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'internal' / 't03_data_integrity_audit.ipynb'
CONFIG_PATH = REPO_ROOT / 'configs' / 't03_audit.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'
STAGE = 't03_pre_split_audit'
POPULATION = 'validated_unsplit_released_rows'
RUN_BOUNDED_RESOURCE_SMOKE = False  # may be enabled only in a separately recorded VERIFY run

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

## 2. Governed input and predecessor lineage

**Question:** Is this exactly the checksum-pinned T01 derivative already characterized by accepted T02 evidence?

The local selector chooses the input; explicit predecessor manifest paths and hashes come from the committed T03 configuration. Filename discovery is never authoritative.

In [2]:
from src.audit import t03_implementation_environment
_base_data_environment = implementation_environment
def implementation_environment():
    evidence = _base_data_environment()
    evidence.update(t03_implementation_environment())
    return evidence

In [3]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['lifecycle_state'] == 'T03_PRE_SPLIT_IN_PROGRESS'
assert config['scope_guards']['construct_outer_split'] is False
assert config['scope_guards']['access_held_out'] is False
assert config['scope_guards']['run_final_calibration'] is False

selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)  # verifies selector-pinned processed SHA-256 before opening
processed_sha256 = selector.payload['processed_sha256']

predecessor_rows = []
for predecessor in config['input']['predecessor_evidence']:
    manifest_path = REPO_ROOT / predecessor['manifest_path']
    observed_hash = sha256_file(manifest_path)
    if observed_hash != predecessor['manifest_sha256']:
        raise AuditContractError(f"Predecessor manifest changed: {predecessor['task']}")
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if not str(manifest['status']).startswith('COMPLETED'):
        raise AuditContractError(f"Predecessor is not complete: {predecessor['task']}")
    processed_links = [a for a in manifest.get('external_artifacts', []) if a.get('role') == 'manifest_selected_processed_derivative']
    if len(processed_links) != 1 or processed_links[0]['sha256'] != processed_sha256:
        raise AuditContractError(f"Processed lineage mismatch: {predecessor['task']}")
    predecessor_rows.append({
        'task': predecessor['task'], 'run_id': predecessor['run_id'],
        'manifest_path': predecessor['manifest_path'], 'manifest_sha256': observed_hash,
        'processed_sha256': processed_links[0]['sha256'], 'status': 'PASS',
    })
predecessor_reconciliation = pd.DataFrame(predecessor_rows)
predecessor_reconciliation

,task,run_id,manifest_path,manifest_sha256,processed_sha256,status
0,T01,t01_production_20260812T091631Z_224672,outputs/runs/t01_production_20260812T091631Z_2...,57e88ed7a2b4690c982460be414bc3c95dc1f73031a066...,fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c...,PASS
1,T02,t02_eda_20260813T094750Z_546902,outputs/runs/t02_eda_20260813T094750Z_546902/a...,b7af9223e9522a77c5e24ff2923ce073651300813a9faf...,fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c...,PASS


## 3. Immutable T03-A run initialization

This cell is the governed writer boundary for this run. It creates a unique run and records configuration, environment, input snapshot, predecessor hashes, and source-only code identity.

In [4]:
RUN_CREATED_AT = utc_now()
RUN_ID = datetime.now(timezone.utc).strftime('t03a_audit_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

run_config = copy.deepcopy(config)
run_config.update({
    'run_id': RUN_ID, 'created_at_utc': RUN_CREATED_AT, 'stage': STAGE,
    'notebook_sources_sha256': notebook_source_sha256(NOTEBOOK_PATH),
    'src_audit_sha256': sha256_file(REPO_ROOT / 'src' / 'audit.py'),
    'git_head': git_head, 'git_dirty': git_dirty,
})
write_t03_json_new(RUN_ROOT, 'audit/run_config.json', run_config)

environment = implementation_environment()
environment.update({'run_id': RUN_ID, 'created_at_utc': RUN_CREATED_AT, 'pandas': pd.__version__, 'nbformat': nbformat.__version__})
write_t03_json_new(RUN_ROOT, 'audit/environment.json', environment)

data_manifest = copy.deepcopy(selector.payload)
data_manifest.update({
    'manifest_role': 'IMMUTABLE_RUN_SNAPSHOT', 'run_id': RUN_ID,
    'resolved_at_utc': RUN_CREATED_AT,
    'selector_path': portable_repo_path(SELECTOR_PATH, REPO_ROOT),
    'raw_path': portable_repo_path(selector.raw_csv_path, REPO_ROOT),
    'processed_path': portable_repo_path(selector.processed_path, REPO_ROOT),
})
write_t03_json_new(RUN_ROOT, 'audit/data_manifest.json', data_manifest)
write_t03_json_new(RUN_ROOT, 'audit/predecessor_reconciliation.json', {'run_id': RUN_ID, 'rows': predecessor_rows})

WindowsPath('C:/Users/phunm5/Documents/criteo_attribution_dataset/outputs/runs/t03a_audit_20260818T072125Z_409014/audit/predecessor_reconciliation.json')

## 4. Schema, semantic roles, source identity, missingness, finite values, and global support

**Method:** explicitly materialize the operation-required pre-split columns, retain their primary `float64` representation, and apply HG/ED checks without filtering. Missing feature values are characterized and preserved; they are not imputed here.

In [5]:
audit_started = time.perf_counter()
process = psutil.Process()
start_rss = process.memory_info().rss

# Explicit operation-specific Pandas materialization: duplicate/SMD checks require joint rows.
frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(frame.columns) != PROCESSED_COLUMNS:
    raise AuditContractError('Processed columns changed during materialization')
if not all(str(frame[feature].dtype) == 'float64' for feature in FEATURE_COLUMNS):
    raise AuditContractError('Primary f0-f11 precision is not float64')

pre_split_integrity = validate_pre_split_frame(frame, require_zero_based_complete=True)
pre_split_integrity.insert(0, 'run_id', RUN_ID)
pre_split_integrity.insert(1, 'population', POPULATION)
write_t03_csv_new(RUN_ROOT, 'audit/pre_split_integrity.csv', pre_split_integrity)

# Visible denominator check: treatment imbalance and outcome rarity are distinct.
ty = frame.groupby(['treatment', 'conversion'], observed=True).size().reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=['treatment', 'conversion']), fill_value=0)
if int(ty.sum()) != len(frame):
    raise AuditContractError('Four-cell T/Y reconstruction does not reconcile')
ty.to_frame('n')

n
treatment conversion          
0         0            2092874
          1               4063
1         0           11845944
          1              36711

## 5. DP-01 pre-split and DP-02 through DP-05

Rows in repeated-value groups, rows beyond the first, group count, and largest group are reported under exact declared columns. Equal values do **not** identify duplicate people; the released-row population is retained unchanged. DP-06 is sealed/deferred because no T05 split exists.

In [6]:
rows_before_duplicate_audit = len(frame)
duplicate_report = duplicate_audit(frame)
duplicate_report.insert(0, 'run_id', RUN_ID)
duplicate_report.insert(1, 'population', POPULATION)
if len(frame) != rows_before_duplicate_audit:
    raise AuditContractError('Duplicate profiling changed the primary population')
write_t03_csv_new(RUN_ROOT, 'audit/duplicate_audit.csv', duplicate_report)

duplicate_rates = duplicate_report[['run_id', 'population', 'definition_id', 'rows_total', 'rows_in_duplicate_groups', 'rows_beyond_first']].copy()
duplicate_rates['duplicate_group_row_fraction'] = duplicate_rates['rows_in_duplicate_groups'] / duplicate_rates['rows_total']
duplicate_rates['beyond_first_fraction'] = duplicate_rates['rows_beyond_first'] / duplicate_rates['rows_total']
write_t03_csv_new(RUN_ROOT, 'audit/duplicate_rates.csv', duplicate_rates)
duplicate_report

,run_id,population,definition_id,columns,row_universe,precision,rows_total,rows_in_duplicate_groups,rows_beyond_first,duplicate_group_count,largest_group,interpretation,row_removal_authorized
0,t03a_audit_20260818T072125Z_409014,validated_unsplit_released_rows,DP-01,[_source_row_id],unsplit_validated_released_rows,float64_primary,13979592,0,0,0,1,equal released-row values are not evidence of ...,False
1,t03a_audit_20260818T072125Z_409014,validated_unsplit_released_rows,DP-02,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",unsplit_validated_released_rows,float64_primary,13979592,2221150,1259545,961605,12,equal released-row values are not evidence of ...,False
2,t03a_audit_20260818T072125Z_409014,validated_unsplit_released_rows,DP-03,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",unsplit_validated_released_rows,float64_primary,13979592,2811714,1626259,1185455,13,equal released-row values are not evidence of ...,False
3,t03a_audit_20260818T072125Z_409014,validated_unsplit_released_rows,DP-04,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",unsplit_validated_released_rows,float64_primary,13979592,2240397,1270251,970146,12,equal released-row values are not evidence of ...,False
4,t03a_audit_20260818T072125Z_409014,validated_unsplit_released_rows,DP-05,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...",unsplit_validated_released_rows,float64_primary,13979592,2240034,1270039,969995,12,equal released-row values are not evidence of ...,False


## 6. DP-07 source/processed precision reconciliation

**Question:** Did conversion or the explicit float32 sensitivity create/change equality?

Source `float64`, explicit `float32`, and processed primary `float64` are compared on identical source ordinals. Float32-only collisions remain sensitivity evidence and cannot redefine the primary representation.

In [7]:
# Explicit full source materialization is operation-specific and is subject to the later VERIFY resource gate.
# DP-07 source baseline uses the accepted T01 CSV parser so that source and processed values are
# comparable by construction, not by parser coincidence: a different float parser differs by ~1 ULP.
reader = open_csv_reader(selector.raw_csv_path)
if not reader.schema.equals(RAW_SCHEMA, check_metadata=False):
    raise AuditContractError('Raw CSV schema differs from the T01 contract')
source_features = (
    pa.Table.from_batches([batch.select(list(FEATURE_COLUMNS)) for batch in reader])
    .to_pandas(split_blocks=True, self_destruct=True)
)
if len(source_features) != len(frame):
    raise AuditContractError('DP-07 source row count differs from the processed population')
source_features[SOURCE_ROW_ID] = np.arange(len(source_features), dtype=np.int64)
processed_features = frame.loc[:, list(FEATURE_COLUMNS) + [SOURCE_ROW_ID]].copy()
dp07 = precision_reconciliation(source_features, processed_features)
write_t03_csv_new(
    RUN_ROOT, 'audit/duplicate_origin/duplicate_origin_summary.csv',
    pd.DataFrame([
        {'run_id': RUN_ID, 'representation': key, **value}
        for key, value in [
            ('source_float64', dp07['source_float64']),
            ('float32_sensitivity', dp07['float32_sensitivity']),
            ('processed_float64', dp07['processed_float64']),
        ]
    ]),
)
write_t03_json_new(RUN_ROOT, 'audit/duplicate_origin/duplicate_origin_report.json', {'run_id': RUN_ID, **dp07})
del source_features, processed_features

## 7. Source/design evidence and ASL limitations

Observed diagnostics cannot prove randomization, pre-treatment timing, consistency, no interference, outcome observability, or transportability. Anonymous features receive no invented semantics, and `_source_row_id` is not a person identifier.

In [8]:
assumption_rows = [
    ('ASL-01', 'LIMITED', 'LIMITATION', 'Exact released-data assignment probabilities and hidden design blocks are unavailable; T03-D02 uses a labelled approximation.'),
    ('ASL-02', 'LIMITED', 'LIMITATION', 'f0-f11 are operationally treated as pre-treatment; complete primary-source timing remains unavailable.'),
    ('ASL-03', 'LIMITED', 'LIMITATION', 'visit remains secondary and exposure audit-only; neither enters X or filters the population.'),
    ('ASL-04', 'LIMITED', 'LIMITATION', 'Observed columns cannot establish consistency or absence of treatment versions.'),
    ('ASL-05', 'LIMITED', 'LIMITATION', 'No public user/network identity supports a complete interference audit.'),
    ('ASL-06', 'LIMITED', 'LIMITATION', 'Outcome-window and measurement evidence remain source limitations.'),
    ('ASL-07', 'LIMITED', 'LIMITATION', 'Claims remain restricted to eligible released rows and represented support.'),
    ('ASL-08', 'SUPPORTED', 'LIMITATION', 'Only one potential outcome is observed; true ITE and empirical PEHE are unavailable.'),
]
assumption_evidence = pd.DataFrame(assumption_rows, columns=['audit_id', 'evidence_status', 'required_action', 'interpretation'])
assumption_evidence.insert(0, 'run_id', RUN_ID)
assumption_evidence.insert(1, 'population', POPULATION)
assumption_evidence['evidence_class'] = 'ASSUMPTION_SUPPORT_OR_LIMITATION'
assumption_evidence['execution_state'] = 'EXECUTED_SOURCE_REVIEW'
write_t03_csv_new(RUN_ROOT, 'audit/assumption_evidence.csv', assumption_evidence)

WindowsPath('C:/Users/phunm5/Documents/criteo_attribution_dataset/outputs/runs/t03a_audit_20260818T072125Z_409014/audit/assumption_evidence.csv')

## 8. Frozen T03-C conditional-permutation and diagnostic contract

Only `T` is permuted. Canonical analytical order is ascending `_source_row_id`; documented blocks use deterministic key order and preserve within-block arm counts. Each assignment rebuilds five treatment-stratified folds and fits five fresh classifiers. Production stores compact assignment/fold hashes, not 2,000 full row-level mappings. `empirical_tail_fraction` is a calibration summary—not a permutation p-value.

In [9]:
calibration_contract = algorithm_contract()
calibration_schema = calibration_artifact_schema()
if calibration_contract['execution_state'] != 'DEFERRED_T03_C':
    raise AuditContractError('Final calibration must remain deferred to T03-C')
write_t03_json_new(RUN_ROOT, 'audit/randomization_calibration/algorithm_contract.json', calibration_contract)
write_t03_json_new(RUN_ROOT, 'audit/randomization_calibration/artifact_schema.json', calibration_schema)

# Formula remains visible: SMD_j=(mean_1-mean_0)/sqrt((var_1(ddof=1)+var_0(ddof=1))/2).
# Operational maximum is max(abs(SMD_j)) over eligible features; |SMD|=.10 is reference-only.
# T03-A computes this observed statistic on the unsplit production population (next section);
# only its INFO/WARNING/MATERIAL_CONCERN disposition is deferred to T03-C's calibrated percentiles.
{'calibration_execution_state': calibration_contract['execution_state'], 'production_thresholds_generated': False}

{'calibration_execution_state': 'DEFERRED_T03_C',
 'production_thresholds_generated': False}

## 9. T03-A.9 — ED-03 observed balance statistics on the full pre-split population

**Question:** What is the observed per-feature standardized mean difference and variance ratio, computed once against every row of the unsplit population?

**Method:** one per-arm count/mean/variance pass over all twelve features (`balance_diagnostics`), reusing the formula frozen above. Each value is a genuinely computed observation, not a placeholder — only its `INFO`/`WARNING`/`MATERIAL_CONCERN` disposition is pending, because that mapping requires the T03-C design-null calibrated percentiles, which do not exist until the 2,000-draw calibration runs on development membership.

In [10]:
balance_table, balance_summary = balance_diagnostics(frame)
balance_table.insert(0, 'run_id', RUN_ID)
balance_table.insert(1, 'population', POPULATION)
write_t03_csv_new(RUN_ROOT, 'audit/balance_diagnostics.csv', balance_table)

ed03_pending = pending_disposition_evidence_record(
    'ED-03',
    observed={
        'max_absolute_smd': balance_summary['max_absolute_smd'],
        'max_feature': balance_summary['max_feature'],
        'eligible_features': balance_summary['eligible_features'],
        'excluded_global_constants': balance_summary['excluded_global_constants'],
    },
)
balance_summary

{'max_absolute_smd': 0.04883634021547789,
 'max_feature': 'f3',
 'eligible_features': ['f0',
  'f1',
  'f2',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9',
  'f10',
  'f11'],
 'excluded_global_constants': [],
 'threshold_source': 'DEFERRED_T03_C_DESIGN_CALIBRATION'}

## 10. Synthetic correctness fixture

This bounded fixture is implementation evidence only. It verifies order invariance, block-count preservation, deterministic stream identity, mapping regeneration, fold isolation, SMD sign, and fresh classifier construction without producing project thresholds or empirical full-data claims. It is executed only in a later VERIFY phase.

In [11]:
fixture = pd.DataFrame({feature: np.arange(40, dtype='float64') + i / 100 for i, feature in enumerate(FEATURE_COLUMNS)})
fixture['treatment'] = np.arange(40) % 2
fixture['conversion'] = (np.arange(40) // 2) % 2
fixture['visit'] = (np.arange(40) + 1) % 2
fixture['exposure'] = np.arange(40) % 2
fixture[SOURCE_ROW_ID] = np.arange(40, dtype='int64')
fixture['fixture_block'] = np.repeat(['A', 'B'], 20)

perm_a, evidence_a = conditional_permute_treatment(fixture, replication_index=0, block_columns=['fixture_block'])
perm_b, evidence_b = conditional_permute_treatment(fixture.sample(frac=1, random_state=123), replication_index=0, block_columns=['fixture_block'])
if evidence_a['assignment_sha256'] != evidence_b['assignment_sha256']:
    raise AuditContractError('Canonical permutation is input-order dependent')
_, stream0 = replication_seed_metadata(0)
_, stream1 = replication_seed_metadata(1)
if stream0['stream_identity_sha256'] == stream1['stream_identity_sha256']:
    raise AuditContractError('Distinct replication indices share a stream identity')

fixture_balance_table, fixture_balance_summary = balance_diagnostics(fixture)
fixture_predictability = cross_fitted_treatment_predictability(fixture)
algorithm_verification = {
    'run_id': RUN_ID, 'execution_mode': 'UNIT_FIXTURE',
    'assignment_hash_reproduced_after_reorder': True,
    'distinct_stream_identities': True,
    'chance_collision_policy': 'VALID_NO_RETRY_OR_DISCARD',
    # bool(...) at the notebook boundary: np.isfinite returns np.bool_, which json.dumps rejects.
    'fixture_max_abs_smd_defined': bool(np.isfinite(fixture_balance_summary['max_absolute_smd'])),
    'fixture_oof_rows': len(fixture_predictability['oof_probability']),
    'fixture_fold_hash': fixture_predictability['fold_evidence']['fold_assignment_sha256'],
    'fixture_fresh_classifier_fits': sum(int(row['fresh_classifier']) for row in fixture_predictability['fold_records']),
    'production_thresholds_generated': False,
    'held_out_access': False,
}
write_t03_json_new(RUN_ROOT, 'audit/randomization_calibration/algorithm_verification.json', algorithm_verification)

WindowsPath('C:/Users/phunm5/Documents/criteo_attribution_dataset/outputs/runs/t03a_audit_20260818T072125Z_409014/audit/randomization_calibration/algorithm_verification.json')

## 11. Bounded resource-smoke interface

The future workload is 2,000 assignments × 5 fresh fits. A resource smoke may estimate feasibility but cannot weaken the replication/fold contract or generate final thresholds. No fixed RAM-percentage rule is used.

In [12]:
resource_smoke = {
    'run_id': RUN_ID,
    'execution_mode': 'SMOKE_ONLY' if RUN_BOUNDED_RESOURCE_SMOKE else 'NON_FINAL_RESOURCE_EVIDENCE',
    'execution_state': 'NOT_EXECUTED_IN_T03A_IMPLEMENT',
    'final_calibration': False, 'replications': 0, 'thresholds_generated': False,
    'worker_rule': 'bounded process pool across replications; LightGBM n_jobs=1 inside each fit',
    'fixed_ram_percentage_gate': False,
}
if RUN_BOUNDED_RESOURCE_SMOKE:
    smoke_started = time.perf_counter()
    smoke = frame.sort_values(SOURCE_ROW_ID, kind='mergesort').head(50_000).copy()
    smoke_mapping, smoke_permutation = conditional_permute_treatment(smoke, replication_index=0)
    smoke = smoke.drop(columns=['treatment']).merge(smoke_mapping, on=SOURCE_ROW_ID, validate='one_to_one')
    smoke_result = cross_fitted_treatment_predictability(smoke, treatment_column='T_perm')
    resource_smoke.update({
        'execution_state': 'EXECUTED_SMOKE_ONLY', 'population': 'canonical_unsplit_50K_prefix',
        'replications': 1, 'fold_fits': 5, 'elapsed_seconds': time.perf_counter() - smoke_started,
        'rss_bytes_observed': process.memory_info().rss,
        'assignment_sha256': smoke_permutation['assignment_sha256'],
        'fold_assignment_sha256': smoke_result['fold_evidence']['fold_assignment_sha256'],
        'metric_names_only': ['roc_auc', 'log_loss', 'constant_prevalence_log_loss', 'log_loss_gain'],
        'result_use': 'NON_FINAL_RESOURCE_EVIDENCE_ONLY',
    })
write_t03_json_new(RUN_ROOT, 'audit/randomization_calibration/resource_smoke.json', resource_smoke)

WindowsPath('C:/Users/phunm5/Documents/criteo_attribution_dataset/outputs/runs/t03a_audit_20260818T072125Z_409014/audit/randomization_calibration/resource_smoke.json')

## 12. Evidence/action separation, interpretation, and lifecycle closure

`evidence_class`, `evidence_status`, `required_action`, and `execution_state` remain distinct. ED-03's observed statistic is executed here; its disposition, and ED-04 entirely, remain null until T03-C. Held-out DP-06 remains sealed. This run may close as T03-A evidence pending review; it must not mark `T03_PRE_SPLIT_COMPLETE` or `T03_ACCEPTED`.

In [13]:
supplemental_audit_rows = pd.DataFrame([
    {
        'run_id': RUN_ID, 'population': POPULATION, 'audit_id': 'HG-01',
        'evidence_class': 'HARD_GATE', 'evidence_status': 'PASS', 'required_action': 'PASS',
        'execution_state': 'EXECUTED', 'passed': True,
        'observed': f"processed_sha256={processed_sha256}; predecessors={len(predecessor_rows)}",
    },
    {
        'run_id': RUN_ID, 'population': POPULATION, 'audit_id': 'ED-02',
        'evidence_class': 'EMPIRICAL_DIAGNOSTIC', 'evidence_status': 'INFO', 'required_action': 'PASS',
        'execution_state': 'EXECUTED', 'passed': True,
        'observed': json.dumps({f'T={t},Y={y}': int(n) for (t, y), n in ty.items()}, sort_keys=True),
    },
    {
        'run_id': RUN_ID, 'population': POPULATION, 'audit_id': 'DP-07',
        'evidence_class': 'EMPIRICAL_DIAGNOSTIC', 'evidence_status': 'INFO', 'required_action': 'PASS',
        'execution_state': 'EXECUTED', 'passed': True,
        'observed': f"source_processed_semantic_equal={dp07['source_processed_semantic_equal']}; float32_additional_rows_beyond_first={dp07['float32_additional_rows_beyond_first']}",
    },
])
pre_split_integrity = pd.concat([pre_split_integrity, supplemental_audit_rows], ignore_index=True, sort=False)

In [14]:
audit_additions = []
for duplicate in duplicate_report.itertuples(index=False):
    repeated = int(duplicate.rows_beyond_first) > 0
    is_identity = duplicate.definition_id == 'DP-01'
    audit_additions.append({
        'run_id': RUN_ID, 'population': POPULATION, 'audit_id': duplicate.definition_id,
        'evidence_class': 'HARD_GATE' if is_identity else 'EMPIRICAL_DIAGNOSTIC',
        'evidence_status': ('FAIL' if repeated else 'PASS') if is_identity else ('WARNING' if repeated else 'INFO'),
        'required_action': ('STOP' if repeated else 'PASS') if is_identity else ('WARNING' if repeated else 'PASS'),
        'execution_state': 'EXECUTED', 'passed': not repeated if is_identity else True,
    })
pre_split_integrity = pd.concat([
    pre_split_integrity, pd.DataFrame(audit_additions),
    assumption_evidence.assign(passed=pd.NA),
], ignore_index=True, sort=False)
# DP-07 is kept in its exact origin artifact; its lineage failure would already have raised.
pre_split_integrity[['audit_id', 'evidence_class', 'evidence_status', 'required_action', 'execution_state']]

,audit_id,evidence_class,evidence_status,required_action,execution_state
0,HG-02,HARD_GATE,PASS,PASS,EXECUTED
1,HG-03,HARD_GATE,PASS,PASS,EXECUTED
2,HG-04,HARD_GATE,PASS,PASS,EXECUTED
3,HG-07,HARD_GATE,PASS,PASS,EXECUTED
4,ED-01-f0,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED
5,ED-01-f1,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED
6,ED-01-f2,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED
7,ED-01-f3,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED
8,ED-01-f4,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED
9,ED-01-f5,EMPIRICAL_DIAGNOSTIC,INFO,PASS,EXECUTED


In [15]:
audit_rows = pre_split_integrity[['run_id', 'population', 'audit_id', 'evidence_class', 'evidence_status', 'required_action', 'execution_state']].copy()
ed03_row = {'evidence_class': 'EMPIRICAL_DIAGNOSTIC', **{
    k: v for k, v in ed03_pending.items() if k in {'audit_id', 'evidence_status', 'required_action', 'execution_state'}
}}
for deferred in [ed03_row, {'evidence_class': 'EMPIRICAL_DIAGNOSTIC', **deferred_evidence_record('ED-04')}, {'evidence_class': 'EMPIRICAL_DIAGNOSTIC', **deferred_evidence_record('DP-06', held_out_sealed=True)}]:
    audit_rows = pd.concat([audit_rows, pd.DataFrame([{
        'run_id': RUN_ID, 'population': POPULATION, **deferred,
    }])], ignore_index=True)
write_t03_csv_new(RUN_ROOT, 'audit/audit_summary.csv', audit_rows)

causal_audit = {
    'run_id': RUN_ID, 'population': POPULATION,
    'estimand': 'assignment_ITT_CATE',
    'treatment': 'treatment', 'primary_outcome': 'conversion',
    'diagnostics_do_not_prove_randomization': True,
    'predicted_uplift_is_not_true_ITE': True,
    'empirical_PEHE_on_real_data': False,
    'held_out_access': False,
}
write_t03_json_new(RUN_ROOT, 'audit/causal_audit.json', causal_audit)

interpretation = pd.DataFrame([
    {
        'run_id': RUN_ID, 'population': POPULATION, 'output_id': 'pre_split_integrity',
        'question': 'Does the manifest-selected unsplit population satisfy mechanical T03-A integrity gates?',
        'method': 'Checksum-pinned T01 loading plus explicit schema/domain/source-ID checks.',
        'evidence': 'audit/pre_split_integrity.csv',
        'interpretation': 'Mechanical results govern whether the affected audit workflow may continue.',
        'what_cannot_be_concluded': 'Passing integrity gates does not prove causal identification or randomization.',
        'required_or_deferred_action': 'Use machine-readable evidence/action fields; stop on any hard-gate failure.',
    },
    {
        'run_id': RUN_ID, 'population': POPULATION, 'output_id': 'duplicate_profiles',
        'question': 'How often do the frozen DP-01 through DP-05 value definitions repeat?',
        'method': 'Exact grouping under named columns at primary float64 precision.',
        'evidence': 'audit/duplicate_audit.csv; audit/duplicate_rates.csv',
        'interpretation': 'Repeated profiles describe released-row values and possible empirical weighting.',
        'what_cannot_be_concluded': 'Equal rows are not duplicate people and same X with different T/Y is not a label conflict.',
        'required_or_deferred_action': 'Retain all released rows; do not deduplicate.',
    },
    {
        'run_id': RUN_ID, 'population': POPULATION, 'output_id': 'ed03_balance_diagnostics',
        'question': 'What is the observed per-feature SMD and variance ratio on the full unsplit population?',
        'method': 'One per-arm count/mean/variance pass over all twelve features (balance_diagnostics).',
        'evidence': 'audit/balance_diagnostics.csv',
        'interpretation': 'Real observed statistics; a large or small value is consistent with, but never proof of, random assignment.',
        'what_cannot_be_concluded': 'No INFO/WARNING/MATERIAL_CONCERN disposition exists until T03-C calibrates the design-null percentiles.',
        'required_or_deferred_action': 'EXECUTED_PENDING_T03C_DISPOSITION with null evidence_status and required_action.',
    },
    {
        'run_id': RUN_ID, 'population': POPULATION, 'output_id': 'randomization_calibration',
        'question': 'Is the final ED-03/ED-04 calibration ready to execute?',
        'method': 'Frozen conditional-permutation algorithm and compact regeneration hashes.',
        'evidence': 'audit/randomization_calibration/algorithm_contract.json',
        'interpretation': 'The method is accepted and implemented, but numerical execution requires T05 development membership.',
        'what_cannot_be_concluded': 'No p95/p99 threshold, causal proof, or final evidence status exists in T03-A.',
        'required_or_deferred_action': 'DEFERRED_T03_C with null evidence_status and required_action.',
    },
])
write_t03_csv_new(RUN_ROOT, 'tables/t03a_interpretation.csv', interpretation)

summary = {
    'run_id': RUN_ID, 'status': 'COMPLETED_T03A_EVIDENCE_PENDING_REVIEW',
    'task_lifecycle_state': 'T03_PRE_SPLIT_IN_PROGRESS',
    'rows_removed': 0, 'outer_split_constructed': False, 'held_out_access': False,
    'final_calibration_executed': False, 'production_null_draws_created': False,
    'outcome_or_uplift_models_trained': False, 't03_accepted': False,
    'next_phase': 'VERIFY_ONLY_AFTER_OWNER_AUTHORIZATION',
    'elapsed_seconds': time.perf_counter() - audit_started,
    'start_rss_bytes': start_rss, 'observed_rss_bytes_at_summary': process.memory_info().rss,
}
write_t03_json_new(RUN_ROOT, 'audit/t03a_summary.json', summary)

manifest_path = finalize_artifact_manifest(
    RUN_ROOT, run_id=RUN_ID, final_status=summary['status'], created_at_utc=utc_now(),
    stage=STAGE, population=POPULATION,
    external_artifacts=[
        {'path': data_manifest['processed_path'], 'role': 'manifest_selected_processed_derivative', 'sha256': processed_sha256, 'status': 'PASS'},
        {'path': 'notebooks/internal/t03_data_integrity_audit.ipynb#sources', 'role': 'human_readable_protocol_source', 'sha256': notebook_source_sha256(NOTEBOOK_PATH), 'status': 'PASS'},
        {'path': 'src/audit.py', 'role': 'reusable_t03_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'audit.py'), 'status': 'PASS'},
    ],
)
summary

{'run_id': 't03a_audit_20260818T072125Z_409014',
 'status': 'COMPLETED_T03A_EVIDENCE_PENDING_REVIEW',
 'task_lifecycle_state': 'T03_PRE_SPLIT_IN_PROGRESS',
 'rows_removed': 0,
 'outer_split_constructed': False,
 'held_out_access': False,
 'final_calibration_executed': False,
 'production_null_draws_created': False,
 'outcome_or_uplift_models_trained': False,
 't03_accepted': False,
 'next_phase': 'VERIFY_ONLY_AFTER_OWNER_AUTHORIZATION',
 'elapsed_seconds': 160.3184159999946,
 'start_rss_bytes': 233889792,
 'observed_rss_bytes_at_summary': 1525125120}

## 13. Deferred work and limitations

- T05 must create and seal the frozen outer split.
- T03-B then establishes split-identity and development-integrity evidence (DP-01/HG-07, ED-02) on train/validation membership.
- T03-C then regenerates canonical development membership and runs the full 2,000-draw calibration, producing the ED-03/ED-04 disposition this run leaves pending.
- DP-06 held-out feature-profile evidence remains `SEALED_DEFERRED_BY_TEST_ISOLATION`.
- Favorable balance or treatment-predictability evidence would remain diagnostic support, never proof of randomization.
- No result here changes the retain-all duplicate policy, frozen feature set, estimand, model roles, or held-out boundary.